## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI:

```bash
az login
```

# 🏦 Custom Chat Message Store for Threads

## Industry Use Case: Compliance-Ready Conversation Audit Trail

This notebook demonstrates how to implement a **custom chat message store** for managing conversation threads with compliance and audit requirements.

| Feature | FSI Application |
|---------|----------------|
| **Custom Storage** | Store conversations in compliance-approved databases |
| **Full Serialization** | Complete message history for regulatory audits |
| **Data Control** | Meet data residency and retention requirements |
| **Flexible Backend** | Integrate with existing enterprise systems |

In [1]:
import os
import json
from typing import Any, Sequence
from azure.identity import AzureCliCredential
from agent_framework import Agent, Message, BaseHistoryProvider, AgentSession
from agent_framework.foundry import FoundryChatClient
from dotenv import load_dotenv

load_dotenv("../../.env")
print("✅ Environment loaded and dependencies imported")

✅ Environment loaded and dependencies imported


## Implement Custom Chat Message Store

This store could integrate with compliance-approved databases (PostgreSQL, Cosmos DB, etc.) for regulatory audit requirements.

In [2]:
class CustomHistoryProvider(BaseHistoryProvider):
    """Custom history provider for compliance-ready conversation storage.
    
    In production, this would integrate with:
    - Relational databases (PostgreSQL, SQL Server)
    - NoSQL databases (Cosmos DB, MongoDB)
    - Enterprise storage systems with audit capabilities
    """

    def __init__(self, source_id: str = "custom-store") -> None:
        super().__init__(source_id)
        self._sessions: dict[str, list[Message]] = {}

    async def get_messages(
        self, session_id: str | None, *, state: dict[str, Any] | None = None, **kwargs: Any
    ) -> list[Message]:
        """Retrieve all messages for a session (would query database in production)."""
        if session_id is None:
            return []
        return list(self._sessions.get(session_id, []))

    async def save_messages(
        self, session_id: str | None, messages: Sequence[Message], *, state: dict[str, Any] | None = None, **kwargs: Any
    ) -> None:
        """Save messages for a session (would write to database in production)."""
        if session_id is None:
            return
        if session_id not in self._sessions:
            self._sessions[session_id] = []
        self._sessions[session_id].extend(messages)


print("✅ CustomHistoryProvider defined")

✅ CustomHistoryProvider defined


## Define Compliance Officer Tools

In [3]:
async def check_transaction_compliance(transaction_id: str) -> str:
    """Check if a transaction meets compliance requirements."""
    compliance_data = {
        "TXN-001": {"status": "Compliant", "risk_level": "Low", "aml_check": "Passed"},
        "TXN-002": {"status": "Under Review", "risk_level": "High", "aml_check": "Pending"},
        "TXN-003": {"status": "Flagged", "risk_level": "Critical", "aml_check": "Failed"},
    }
    c = compliance_data.get(transaction_id, {"status": "Unknown", "risk_level": "Unknown", "aml_check": "Unknown"})
    return f"Transaction {transaction_id}: Status={c['status']}, Risk={c['risk_level']}, AML Check={c['aml_check']}"


async def get_regulatory_requirements(regulation_type: str) -> str:
    """Get regulatory requirements for a specific regulation."""
    regulations = {
        "AML": "Anti-Money Laundering: Customer verification, transaction monitoring, suspicious activity reporting",
        "KYC": "Know Your Customer: Identity verification, risk assessment, ongoing monitoring",
        "GDPR": "Data Protection: Consent management, data minimization, right to erasure",
    }
    return regulations.get(regulation_type, f"No specific requirements found for {regulation_type}")


print("✅ Tools defined: check_transaction_compliance, get_regulatory_requirements")

✅ Tools defined: check_transaction_compliance, get_regulatory_requirements


## Run Compliance Officer with Custom Store

The agent uses `chat_message_store_factory` to create custom store instances for each thread.

In [4]:
async def run_compliance_officer():
    """Run compliance officer with custom history provider."""
    print("\n--- Compliance Officer with Custom History Provider ---\n")
    
    # Note: Custom history providers work with in-memory chat clients
    # FoundryChatClient uses Azure AI Foundry with local message management
    PROJECT_ENDPOINT = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")
    
    print(f"Using project endpoint: {PROJECT_ENDPOINT}")
    print(f"Using deployment: {MODEL_DEPLOYMENT}\n")
    
    # Create the custom history provider instance
    history_provider = CustomHistoryProvider()
    
    client = FoundryChatClient(
        project_endpoint=PROJECT_ENDPOINT,
        model=MODEL_DEPLOYMENT,
        credential=AzureCliCredential(),
    )
    agent = Agent(
        client=client,
        name="ComplianceOfficer",
        instructions="You are a compliance officer helping with regulatory inquiries. Check transactions and provide regulatory guidance.",
        tools=[check_transaction_compliance, get_regulatory_requirements],
        context_providers=[history_provider],  # Use custom history provider
    )
    
    # Start new session (creates a conversation session)
    session = agent.create_session()
    print(f"✅ Session created: {session.session_id}\n")
    
    # Phase 1: Initial conversation
    print("=" * 60)
    print("Phase 1: Compliance Review Session")
    print("=" * 60)
    
    query1 = "Please check compliance status for transaction TXN-002"
    print(f"Analyst: {query1}")
    response1 = await agent.run(query1, session=session)
    print(f"Officer: {response1.text}\n")
    
    # Phase 2: Serialize session
    print("=" * 60)
    print("Phase 2: Saving Compliance Session")
    print("=" * 60)
    
    serialized_session = session.to_dict()
    
    print(f"\n💾 Serialized session: {json.dumps(serialized_session, indent=2, default=str)}\n")
    print("📊 Note: Custom history providers maintain full message history")
    print("💡 This enables complete audit trails for compliance\n")
    
    # Phase 3: Resume session
    print("=" * 60)
    print("Phase 3: Resuming Compliance Session")
    print("=" * 60)
    
    resumed_session = AgentSession.from_dict(serialized_session)
    print(f"\n✅ Session restored: {resumed_session.session_id}\n")
    
    query2 = "What do you remember about me?"
    print(f"Analyst: {query2}")
    response2 = await agent.run(query2, session=resumed_session)
    print(f"Officer: {response2.text}\n")
    
    print("=" * 60)
    print("✅ Custom history provider demo completed!")
    print("=" * 60)

await run_compliance_officer()


--- Compliance Officer with Custom History Provider ---

Using project endpoint: https://demopocaifoundry.services.ai.azure.com/api/projects/demoproject
Using deployment: gpt-4o

✅ Session created: 0dc1db7d-07dc-4ed0-b1ee-4a3f4f267695

Phase 1: Compliance Review Session
Analyst: Please check compliance status for transaction TXN-002
Officer: The compliance status for transaction **TXN-002** is currently:

- **Status:** Under Review  
- **Risk Level:** High  
- **AML Check:** Pending  

Further checks or approvals may be necessary before proceeding. Let me know if you need detailed regulatory guidance or actions for high-risk transactions.

Phase 2: Saving Compliance Session

💾 Serialized session: {
  "type": "session",
  "session_id": "0dc1db7d-07dc-4ed0-b1ee-4a3f4f267695",
  "service_session_id": "resp_0e3aede0fccdc0580069d850003b308196af5c65ecf5e980a1",
  "state": {
    "custom-store": {}
  }
}

📊 Note: Custom history providers maintain full message history
💡 This enables complete a

## Key Takeaways

| Feature | Description |
|---------|-------------|
| `BaseHistoryProvider` | Base class for custom conversation storage backends |
| `context_providers` | Agent parameter to attach history providers |
| `AgentSession` | Session object with `to_dict()` / `from_dict()` for serialization |

## Production Considerations

| Aspect | Implementation |
|--------|---------------|
| **Database** | PostgreSQL, Cosmos DB, SQL Server |
| **Audit Trail** | Immutable logs with timestamps |
| **Encryption** | Encrypt sensitive conversation data |
| **Retention** | Implement data retention policies |